# 02 - Carga del Anuario de Aforos

Este notebook carga los datos del Anuario de Aforos del MITECO (CEDEX) correspondientes a la Demarcación Hidrográfica del Miño-Sil.

## Proceso

1. Verificación del alcance geográfico de la descarga.
2. Catálogo de embalses y cruce con el maestro de embalses del SAIH.
3. Carga de datos diarios de embalses (AFLIQE).
4. Carga de estaciones de aforo en río y sus datos diarios (AFLIQ).
5. Carga de estaciones evaporimétricas y datos de evaporación (EVAP).
6. Filtrado por el periodo de estudio (2000-2023) y guardado en `data/interim/`.

## Nota sobre el alcance

La descarga del Anuario se realizó acotada a la Demarcación del Miño-Sil, por lo que los ficheros no contienen datos de otras cuencas y no se requiere filtrado geográfico.
El notebook verifica esta condición de forma explícita.

## Fuente

Anuario de Aforos - Sistema de Información CEDEX  
https://www.miteco.gob.es/es/cartografia-y-sig/ide/descargas/agua/anuario-de-aforos.html

## 1. Imports y configuración

In [1]:
from pathlib import Path
import pandas as pd

# Rutas
DIR_RAW_ANUARIO = Path("../data/raw/anuario")
DIR_PROCESSED = Path("../data/processed")
DIR_INTERIM = Path("../data/interim")
DIR_INTERIM.mkdir(parents=True, exist_ok=True)

# Periodo de estudio
FECHA_INICIO = pd.Timestamp("2000-01-01")
FECHA_FIN = pd.Timestamp("2023-12-31")

# Parámetros de lectura comunes de los CSV del Anuario
CSV_KWARGS = {"sep": ";", "encoding": "latin-1"}

# Diccionario de tipos de dato de embalse (reemplaza el fichero de catálogo ausente)
TIPO_DATO_EMB = {1: "RESERVA (HM3)", 2: "SALIDA (HM3)", 3: "ENTRADA (HM3)"}

# Sistemas de referencia
# El Anuario publica coordenadas en huso 30; SAIH y CHMS trabajan en huso 29.
CRS_ANUARIO = "EPSG:25830"
CRS_PROYECTO = "EPSG:25829"

# Umbral de distancia para validar una correspondencia entre fuentes (metros).
# Las correspondencias válidas quedan por debajo de 400 m y la primera descartada
# supera los 3.500 m, por lo que 1.000 m separa ambos grupos con holgura.
UMBRAL_DIST_M = 1000

## 2. Verificación del alcance geográfico

La descarga del Anuario se solicitó acotada a la Demarcación del Miño-Sil. Se verifica que efectivamente los ficheros contienen una única cuenca antes de continuar, de modo que el notebook falle de forma explícita si en el futuro se utiliza sobre una descarga
de ámbito nacional.

In [2]:
gr_cuenca = pd.read_csv(DIR_RAW_ANUARIO / "gr_cuenca.csv", **CSV_KWARGS)
gr_cuenca["gr_cuenca"] = gr_cuenca["gr_cuenca"].str.strip()

assert len(gr_cuenca) == 1, (
    f"Se esperaba una única cuenca en la descarga, encontradas {len(gr_cuenca)}: "
    f"{gr_cuenca['gr_cuenca'].tolist()}. Revisar el alcance de la descarga."
)

print("=== Alcance de la descarga ===")
print(f"Cuenca: {gr_cuenca['gr_cuenca'].iloc[0]} (id={gr_cuenca['gr_cuenca_id'].iloc[0]})")

=== Alcance de la descarga ===
Cuenca: MIÑO-SIL (id=18)


In [3]:
# Cargar catálogo de embalses y ver de qué cuencas son
embalses = pd.read_csv(DIR_RAW_ANUARIO / "embalse.csv", sep=";", encoding="latin-1")
print(f"Total embalses en el CSV: {len(embalses)}")
print(f"\nColumnas: {list(embalses.columns)}")
print(f"\nMuestra:")
embalses.head()

Total embalses en el CSV: 35

Columnas: ['ref_ceh', 'nom_embalse', 'serv', 'comentario', 'muni_id', 'hoja_id', 'xutm', 'yutm', 'xutm30', 'yutm30', 'long', 'lat', 'longwgs84', 'latwgs84', 'mna', 'mnne', 'num_cuenca', 'nae', 'naem', 'xetrs89', 'yetrs89', 'cod_saih']

Muestra:


,ref_ceh,nom_embalse,serv,comentario,muni_id,hoja_id,xutm,yutm,xutm30,yutm30,...,longwgs84,latwgs84,mna,mnne,num_cuenca,nae,naem,xetrs89,yetrs89,cod_saih
0,1627,BELESAR,1,LOS DATOS DESDE 1962 EN ADELANTE ESTAN EN PROC...,27016,155,605695,4720600,113659,4730564,...,-74245,423743,330.0,330.0,1408,59,60,113548,4730358,E001
1,1629,"PEARES, LOS",1,RESERVA MODIFICADA DESDE 1955 (REFERENCIA RESP...,27041,188,605020,4702424,111695,4712451,...,-74327,422754,194.2,194.2,1410,64,67,111584,4712244,E002
2,1631,VELLE,1,LOS DATOS DE SALIDAS DEL 10/2000 AL 01/2003 SU...,32054,188,594674,4690377,100534,4701144,...,-75105,422129,108.4,108.4,1467,52,52,100423,4700937,E030
3,1634,CASTRELO,1,LOS DATOS DE SALIDAS DEL 10/2000 AL 01/2003 SU...,32069,225,572962,4682849,78272,4695155,...,-80659,421733,88.0,88.0,1473,52,52,78160,4694948,E031
4,1637,ALBARELLOS,1,LOS DATOS DE SALIDAS DEL 10/2000 AL 01/2003 SO...,32040,186,566619,4694677,72768,4707428,...,-81131,422359,0.0,265.0,1476,49,50,72656,4707221,E032


## 3. Cruce del catálogo de embalses con el maestro del SAIH

El fichero `embalse.csv` incluye la columna `cod_saih`, pero la codificación difiere de
la utilizada por el SAIH: el Anuario emplea el formato `E` + tres dígitos (`E007`, `E016`)
mientras que el SAIH utiliza `E` + dos dígitos + letra (`E07A`, `E16A`). El cruce se
realiza por tanto sobre la parte numérica del código.

Dado que la parte numérica puede generar correspondencias ambiguas (varios registros
comparten número), cada candidato se valida midiendo la distancia entre las coordenadas
de ambas fuentes, previa reproyección del Anuario al huso 29. Solo se aceptan las
correspondencias por debajo del umbral de distancia establecido.

In [4]:
embalses = pd.read_csv(DIR_RAW_ANUARIO / "embalse.csv", **CSV_KWARGS)

print(f"Embalses en el Anuario: {len(embalses)}")
print(f"Con código SAIH asignado: {embalses['cod_saih'].notna().sum()}")
embalses[["ref_ceh", "nom_embalse", "cod_saih", "mnne", "mna", "nae"]].head()

Embalses en el Anuario: 35
Con código SAIH asignado: 34


,ref_ceh,nom_embalse,cod_saih,mnne,mna,nae
0,1627,BELESAR,E001,330.0,330.0,59
1,1629,"PEARES, LOS",E002,194.2,194.2,64
2,1631,VELLE,E030,108.4,108.4,52
3,1634,CASTRELO,E031,88.0,88.0,52
4,1637,ALBARELLOS,E032,265.0,0.0,49


In [5]:
import re
import numpy as np
from pyproj import Transformer

def num_saih(codigo):
    """Parte numérica de un código SAIH: 'E007' -> 7, 'E07A' -> 7, 'E003-E005' -> 3."""
    m = re.match(r"^E(\d+)", str(codigo).strip().upper())
    return int(m.group(1)) if m else np.nan

maestro = pd.read_parquet(DIR_PROCESSED / "maestro_embalses.parquet")

embalses["num_saih"] = embalses["cod_saih"].apply(num_saih)
maestro["num_saih"] = maestro["ID_SAIH"].apply(num_saih)

# Candidatos: todas las combinaciones que comparten parte numérica
candidatos = embalses[["ref_ceh", "nom_embalse", "cod_saih", "num_saih", "xetrs89", "yetrs89"]].merge(
    maestro[["ID_SAIH", "Nombre_SAIH", "X", "Y", "num_saih"]],
    on="num_saih",
    how="inner",
)

# Reproyección del Anuario al sistema del proyecto y cálculo de distancias
transformer = Transformer.from_crs(CRS_ANUARIO, CRS_PROYECTO, always_xy=True)
x29, y29 = transformer.transform(candidatos["xetrs89"].values, candidatos["yetrs89"].values)
candidatos["dist_m"] = np.hypot(x29 - candidatos["X"], y29 - candidatos["Y"])

# Se aceptan los candidatos bajo umbral; ante duplicados, se conserva el más cercano
validos = candidatos[candidatos["dist_m"] <= UMBRAL_DIST_M].copy()
crosswalk = (
    validos.sort_values("dist_m")
    .drop_duplicates(subset="ref_ceh", keep="first")
    .drop_duplicates(subset="ID_SAIH", keep="first")
    .sort_values("ID_SAIH")
    .reset_index(drop=True)
)

print(f"Correspondencias validadas: {len(crosswalk)}")
print(f"Distancia mediana: {crosswalk['dist_m'].median():.1f} m")
print(f"Distancia máxima:  {crosswalk['dist_m'].max():.1f} m")

assert crosswalk["dist_m"].max() <= UMBRAL_DIST_M, "Correspondencia fuera de umbral"
assert crosswalk["dist_m"].median() < 200, "Distancia mediana anómala: revisar el CRS de origen"

Correspondencias validadas: 33
Distancia mediana: 85.3 m
Distancia máxima:  395.9 m


In [6]:
#Registros sin correspondencia
sin_saih = embalses[~embalses["ref_ceh"].isin(crosswalk["ref_ceh"])]
sin_anuario = maestro[~maestro["ID_SAIH"].isin(crosswalk["ID_SAIH"])]

print("=== Registros del Anuario sin correspondencia en el SAIH ===")
print(sin_saih[["ref_ceh", "nom_embalse", "cod_saih"]].to_string(index=False))

print("\n=== Embalses del SAIH sin correspondencia en el Anuario ===")
print(sin_anuario[["ID_SAIH", "Nombre_SAIH"]].to_string(index=False))

crosswalk.to_parquet(DIR_PROCESSED / "crosswalk_anuario_saih.parquet", index=False)
print(f"\nTabla de correspondencia guardada: {len(crosswalk)} registros")

=== Registros del Anuario sin correspondencia en el SAIH ===
 ref_ceh          nom_embalse cod_saih
    1781         EDRADA-CONSO    E016B
    1794 VILL / MAO /REGUEIRO      NaN

=== Embalses del SAIH sin correspondencia en el Anuario ===
ID_SAIH            Nombre_SAIH
   E05A    Matalavilla - Presa
   E16D As Portas - C.H. Conso

Tabla de correspondencia guardada: 33 registros


## 4. Datos diarios de embalses (AFLIQE)

La tabla AFLIQE contiene la reserva y la salida diarias de cada embalse. Se filtra únicamente por el periodo de estudio, ya que la descarga solo contiene embalses de la
Demarcación.

In [7]:
afliqe = pd.read_csv(DIR_RAW_ANUARIO / "afliqe.csv", **CSV_KWARGS)
print(f"Registros AFLIQE totales: {len(afliqe):,}")
print(f"Columnas: {list(afliqe.columns)}")

afliqe["fecha"] = pd.to_datetime(afliqe["fecha"], format="%d/%m/%Y", errors="coerce")
n_nat = afliqe["fecha"].isna().sum()
if n_nat:
    print(f"AVISO: {n_nat:,} fechas no parseadas. Revisar el formato de fecha del fichero.")

# Comprobación de integridad: todas las referencias deben estar en el catálogo
refs_catalogo = set(embalses["ref_ceh"])
refs_huerfanas = set(afliqe["ref_ceh"]) - refs_catalogo
if refs_huerfanas:
    print(f"AVISO: {len(refs_huerfanas)} ref_ceh en AFLIQE sin entrada en el catálogo: {refs_huerfanas}")

afliqe_periodo = afliqe[
    afliqe["fecha"].between(FECHA_INICIO, FECHA_FIN)
].copy()

print(f"\nRegistros en el periodo {FECHA_INICIO.year}-{FECHA_FIN.year}: {len(afliqe_periodo):,}")
print(f"Embalses con datos: {afliqe_periodo['ref_ceh'].nunique()}")
print(f"Rango de fechas: {afliqe_periodo['fecha'].min():%Y-%m-%d} a {afliqe_periodo['fecha'].max():%Y-%m-%d}")

Registros AFLIQE totales: 673,779
Columnas: ['ref_ceh', 'fecha', 'reserva', 'salida', 'tipo']

Registros en el periodo 2000-2023: 269,944
Embalses con datos: 34
Rango de fechas: 2000-01-01 a 2021-09-30


## 5. Estaciones de aforo en río y datos diarios (AFLIQ)
La tabla AFLIQ contiene el caudal medio diario de cada estación de aforo en río. Filtramos análogamente.

In [8]:
estaf = pd.read_csv(DIR_RAW_ANUARIO / "estaf.csv", **CSV_KWARGS)

print(f"Estaciones de aforo en el Anuario: {len(estaf)}")
print(f"Con código SAIH asignado: {estaf['cod_saih'].notna().sum()}")
estaf[["indroea", "lugar", "num_cuenca", "cod_saih"]].head(10)

Estaciones de aforo en el Anuario: 90
Con código SAIH asignado: 60


,indroea,lugar,num_cuenca,cod_saih
0,1010,PUENTE QUEROL,1414,NaN
1,1011,RUAPETIN,1436,NaN
2,1012,VILLAFRANCA DEL BIERZO,1426,NaN
3,1018,PUENTE QUIROGA,1454,NaN
4,1019,PACIOS DE VEIGA,1463,NaN
5,1603,REGUNTILLE,1375,A002
6,1605,PONTEVILAR,1378,A003
7,1607,CELA (MARGEN DERECHA),1382,A004D
8,1608,CELA (MARGEN IZQUIERDA),1382,A004I
9,1609,RABADE,1382,A004


In [9]:
afliq = pd.read_csv(DIR_RAW_ANUARIO / "afliq.csv", **CSV_KWARGS)
print(f"Registros AFLIQ totales: {len(afliq):,}")

afliq["fecha"] = pd.to_datetime(afliq["fecha"], format="%d/%m/%Y", errors="coerce")
n_nat = afliq["fecha"].isna().sum()
if n_nat:
    print(f"AVISO: {n_nat:,} fechas no parseadas.")

afliq_periodo = afliq[afliq["fecha"].between(FECHA_INICIO, FECHA_FIN)].copy()

print(f"\nRegistros en el periodo: {len(afliq_periodo):,}")
print(f"Estaciones con datos: {afliq_periodo['indroea'].nunique()}")
print(f"Rango de fechas: {afliq_periodo['fecha'].min():%Y-%m-%d} a {afliq_periodo['fecha'].max():%Y-%m-%d}")

Registros AFLIQ totales: 718,088

Registros en el periodo: 289,588
Estaciones con datos: 62
Rango de fechas: 2000-01-01 a 2021-09-30


## 6. Estaciones evaporimétricas y datos de evaporación (EVAP)
La tabla EVAP contiene la evaporación mensual, útil para calcular el balance hídrico y como variable predictora complementaria.

In [10]:
estev = pd.read_csv(DIR_RAW_ANUARIO / "estev.csv", **CSV_KWARGS)
print(f"Estaciones evaporimétricas: {len(estev)}")

evap = pd.read_csv(DIR_RAW_ANUARIO / "evap.csv", **CSV_KWARGS)
print(f"Registros EVAP totales: {len(evap):,}")

# anomes viene como entero AAAAMM
evap["fecha"] = pd.to_datetime(evap["anomes"].astype(str), format="%Y%m", errors="coerce")
evap_periodo = evap[evap["fecha"].between(FECHA_INICIO, FECHA_FIN)].copy()

print(f"Registros en el periodo: {len(evap_periodo):,}")
print(f"Rango: {evap_periodo['fecha'].min():%Y-%m} a {evap_periodo['fecha'].max():%Y-%m}")

Estaciones evaporimétricas: 5
Registros EVAP totales: 944
Registros en el periodo: 178
Rango: 2000-01 a 2007-06


## 7. Guardado en `data/interim/`

Guardamos los datos filtrados en formato parquet para su uso posterior en el pipeline.

In [11]:
print("=== Cobertura temporal de las series del Anuario ===")
for nombre, df in [("AFLIQE (embalses)", afliqe_periodo),
                   ("AFLIQ (aforo en río)", afliq_periodo),
                   ("EVAP (evaporación)", evap_periodo)]:
    print(f"{nombre:<24} {df['fecha'].min():%Y-%m-%d} a {df['fecha'].max():%Y-%m-%d}")

print(f"\nPeriodo de estudio solicitado: {FECHA_INICIO:%Y-%m-%d} a {FECHA_FIN:%Y-%m-%d}")
print("NOTA: las series del Anuario finalizan en 2020, por lo que no cubren el "
      "periodo reservado para test (2022-2023).")

=== Cobertura temporal de las series del Anuario ===
AFLIQE (embalses)        2000-01-01 a 2021-09-30
AFLIQ (aforo en río)     2000-01-01 a 2021-09-30
EVAP (evaporación)       2000-01-01 a 2007-06-01

Periodo de estudio solicitado: 2000-01-01 a 2023-12-31
NOTA: las series del Anuario finalizan en 2020, por lo que no cubren el periodo reservado para test (2022-2023).


In [12]:
salidas = {
    "anuario_embalses_catalogo.parquet": embalses,
    "anuario_estaciones_aforo_catalogo.parquet": estaf,
    "anuario_estaciones_evap_catalogo.parquet": estev,
    "anuario_afliqe.parquet": afliqe_periodo,
    "anuario_afliq.parquet": afliq_periodo,
    "anuario_evap.parquet": evap,
}

for nombre, df in salidas.items():
    ruta = DIR_INTERIM / nombre
    df.to_parquet(ruta, index=False)
    print(f"  {nombre}: {len(df):,} filas")

print(f"\nGuardado en {DIR_INTERIM}")

  anuario_embalses_catalogo.parquet: 35 filas
  anuario_estaciones_aforo_catalogo.parquet: 90 filas
  anuario_estaciones_evap_catalogo.parquet: 5 filas
  anuario_afliqe.parquet: 269,944 filas
  anuario_afliq.parquet: 289,588 filas
  anuario_evap.parquet: 944 filas

Guardado en ..\data\interim


## 8. Resumen

- La descarga del Anuario está acotada a la Demarcación del Miño-Sil, verificado sobre
  el catálogo de grandes cuencas.
- Catálogo de embalses cruzado con el maestro del SAIH mediante el campo `cod_saih`,
  lo que permite identificar la cobertura adicional que aporta el Anuario.
- Series diarias de embalses (AFLIQE) y de estaciones de aforo en río (AFLIQ) filtradas
  al periodo 2000-2023.
- Catálogo de estaciones evaporimétricas y series de evaporación mensual.

Los ficheros quedan disponibles en `data/interim/` para su uso en el resto del pipeline.